# E-Commerce Customer Support: NLP Classification

## Phase 1 Data Foundations & Exploration (Days 1–4)

### 1.1 Dataset Loading + Repo and project structure set up (Day 1)

We use the [`NebulaByte/E-Commerce_Customer_Support_Conversations`](https://huggingface.co/datasets/NebulaByte/E-Commerce_Customer_Support_Conversations)
dataset from Hugging Face — 1,000 synthetically generated customer support
conversations for an e-commerce platform ("BrownBox"), each labeled with an
issue area/category/sub-category, customer sentiment, product category, and
agent experience level.

We load it via the `datasets` library rather than downloading a raw file
manually, since it handles caching and format conversion (parquet → pandas)
for us and keeps the notebook reproducible for anyone re-running it.

In [1]:
from datasets import load_dataset
import pandas as pd

# Load the dataset directly from the Hugging Face Hub
ds = load_dataset("NebulaByte/E-Commerce_Customer_Support_Conversations")

# Convert the train split to a pandas DataFrame for exploration
df = ds["train"].to_pandas()

print(f"Shape: {df.shape}")
print(f"\nColumns and dtypes:\n{df.dtypes}")

Shape: (1000, 11)

Columns and dtypes:
issue_area                     str
issue_category                 str
issue_sub_category             str
issue_category_sub_category    str
customer_sentiment             str
product_category               str
product_sub_category           str
issue_complexity               str
agent_experience_level         str
agent_experience_level_desc    str
conversation                   str
dtype: object


### Sample rows

A quick look at the raw structure before any cleaning.

In [2]:
df.sample(5, random_state=42)

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation
521,Shipping,Contacting Seller's Partnered Courier Service ...,How to get in touch with the courier service p...,Contacting Seller's Partnered Courier Service ...,neutral,Men/Women/Kids,Pram/Stroller,less,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...
737,Cancellations and returns,Return Checks and Fees,Determination of the Return Fee,Return Checks and Fees -> Determination of the...,negative,Appliances,Coffee Maker,medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for contacting BrownBox custo...
740,Warranty,Lost or Missing Warranty Card,Process of claiming warranty without a warrant...,Lost or Missing Warranty Card -> Process of cl...,frustrated,Appliances,Vacuum Cleaner,medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...
660,Cancellations and returns,Return and Exchange,Eligibility Disputes,Return and Exchange -> Eligibility Disputes,negative,Men/Women/Kids,Shoes,medium,junior,"handles customer inquiries independently, poss...","Customer: Hi, I have an issue with my recent p..."
411,Order,Order Delivery Issues,Inability to track the order,Order Delivery Issues -> Inability to track th...,negative,Electronics,External Hard Disk,medium,experienced,"confidently handles complex customer issues, e...","Customer: Hi, I'm calling because I'm unable t..."


### 1.2 Data Quality Inspection (Day 2)

Before any modeling, we check the dataset for the usual suspects: missing
values, duplicate rows, and inconsistent text formatting (stray whitespace,
inconsistent casing, encoding artifacts). This dataset is synthetically
generated (GPT-3.5), so we don't expect messy real-world issues like typos
or mixed encodings — but we check systematically rather than assuming.

In [3]:
# --- Missing values ---
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({"nulls": null_counts, "pct": null_pct})
print("Missing values per column:")
print(null_summary[null_summary["nulls"] > 0] if null_summary["nulls"].sum() > 0
      else "None found.")

# --- Full-row duplicates ---
n_dupe_rows = df.duplicated().sum()
print(f"\nFully duplicated rows: {n_dupe_rows}")

# --- Duplicate conversations (same text, possibly different labels) ---
n_dupe_convos = df.duplicated(subset=["conversation"]).sum()
print(f"Duplicate 'conversation' text (regardless of other columns): {n_dupe_convos}")

Missing values per column:
None found.

Fully duplicated rows: 0
Duplicate 'conversation' text (regardless of other columns): 2


In [4]:
# --- Check categorical columns for inconsistent casing/whitespace ---
categorical_cols = [
    "issue_area", "issue_category", "issue_sub_category",
    "customer_sentiment", "product_category", "product_sub_category",
    "issue_complexity", "agent_experience_level",
]

for col in categorical_cols:
    raw_values = df[col].unique()
    stripped_lower = set(v.strip().lower() for v in raw_values)
    if len(raw_values) != len(stripped_lower):
        print(f"⚠️  {col}: inconsistent casing/whitespace detected "
              f"({len(raw_values)} raw vs {len(stripped_lower)} normalized)")
    # Flag any leading/trailing whitespace even if it doesn't create a dupe
    has_whitespace_issue = any(v != v.strip() for v in raw_values)
    if has_whitespace_issue:
        print(f"⚠️  {col}: contains values with leading/trailing whitespace")

print("\nIf nothing printed above, categorical columns are clean.")

# --- Check the conversation text for encoding artifacts ---
# Look for common mojibake / non-standard characters
import re
suspicious_pattern = re.compile(r"[Ã¢â‚¬â€œâ€™]|\\x[0-9a-fA-F]{2}")
n_suspicious = df["conversation"].apply(lambda t: bool(suspicious_pattern.search(t))).sum()
print(f"\nConversations with likely encoding artifacts: {n_suspicious}")

# Check for excessive/irregular whitespace in conversation text
n_multi_space = df["conversation"].apply(lambda t: "  " in t).sum()
print(f"Conversations with double spaces: {n_multi_space}")

⚠️  issue_sub_category: contains values with leading/trailing whitespace

If nothing printed above, categorical columns are clean.

Conversations with likely encoding artifacts: 0
Conversations with double spaces: 0


### Data Quality Notes (Day 2)

- **Missing values:** None found across any of the 11 columns.
- **Duplicate rows:** 0 fully duplicated rows.
- **Duplicate conversation text:** 2 conversations share identical text
  despite the row not being a full duplicate — i.e. the same conversation
  appears under (likely) different label combinations. We keep these rather
  than dropping them, since they may reflect legitimate ambiguity in how a
  conversation could be labeled, but we flag this as a potential source of
  train/test leakage if a duplicated conversation ends up split across both
  sets later.
- **Categorical consistency:** `issue_sub_category` had leading/trailing
  whitespace on some values; all other categorical columns were clean.
  Whitespace was stripped in the cleaning pass below.
- **Encoding artifacts:** None found in `conversation` text.
- **Whitespace irregularities:** No double-spaces found in `conversation` text.

Overall, this synthetic dataset is clean as expected — the only real issue
was whitespace in one categorical column, now fixed.

In [5]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Strip whitespace on all string-like columns (works across pandas 2/3)
    str_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        df[col] = df[col].str.strip()

    df["conversation"] = df["conversation"].str.replace(r"\s+", " ", regex=True)

    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    after = len(df)
    if before != after:
        print(f"Dropped {before - after} fully duplicated rows.")

    return df

df_clean = clean_dataframe(df)
print(f"Cleaned shape: {df_clean.shape}")

df_clean.to_parquet("../data/conversations_clean.parquet", index=False)
print("Saved to data/conversations_clean.parquet")

Cleaned shape: (1000, 11)
Saved to data/conversations_clean.parquet


### 1.3 Dataset Exploration: Class Distribution (Day 3)

We're using `issue_area` as our primary classification target — it's the broadest 
categorical label in the dataset and keeps the number of classes manageable for a 
classic ML pipeline (vs. `issue_category` or `issue_sub_category`, which are far more 
granular and would likely produce many classes with very few examples each).

Below we check the frequency of each class to identify any class imbalance, which 
will inform our modeling and evaluation choices later (e.g. whether accuracy alone 
is a meaningful metric).

In [6]:
import plotly.express as px

# Count how many rows fall into each issue_area
class_counts = df['issue_area'].value_counts()
print(class_counts)
print(f"\nNumber of classes: {class_counts.shape[0]}")
print(f"Largest class: {class_counts.idxmax()} ({class_counts.max()} rows, "
      f"{class_counts.max() / len(df):.1%})")
print(f"Smallest class: {class_counts.idxmin()} ({class_counts.min()} rows, "
      f"{class_counts.min() / len(df):.1%})")

# Plot it
fig = px.bar(
    x=class_counts.index,
    y=class_counts.values,
    labels={'x': 'Issue Area', 'y': 'Number of Tickets'},
    title='Class Distribution: issue_area',
    color_discrete_sequence=['steelblue']
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

issue_area
Cancellations and returns    286
Order                        270
Login and Account            151
Shopping                     116
Warranty                     105
Shipping                      72
Name: count, dtype: int64

Number of classes: 6
Largest class: Cancellations and returns (286 rows, 28.6%)
Smallest class: Shipping (72 rows, 7.2%)


### 1.4 Dataset Exploration: Text Length Distribution

The `conversation` column contains full Agent/Customer transcripts. We check the 
distribution of conversation length (in words) to understand typical ticket size 
and spot any extreme outliers (e.g. very short or truncated conversations) that 
may need special handling during preprocessing.

In [7]:
import plotly.graph_objects as go

# Word count per conversation
df['conversation_word_count'] = df['conversation'].str.split().str.len()

print(df['conversation_word_count'].describe())

median_wc = df['conversation_word_count'].median()

fig = px.histogram(
    df, x='conversation_word_count', nbins=50,
    title='Distribution of Conversation Length (word count)',
    labels={'conversation_word_count': 'Word Count'},
    color_discrete_sequence=['steelblue']
)
fig.update_traces(marker_line_color='white', marker_line_width=1)
fig.add_vline(
    x=median_wc, line_dash='dash', line_color='red',
    annotation_text=f'Median: {median_wc:.0f}', annotation_position='top'
)
fig.update_layout(yaxis_title='Number of Tickets')
fig.show()

count    1000.00000
mean      371.04500
std        99.58699
min         8.00000
25%       306.00000
50%       358.00000
75%       424.00000
max       992.00000
Name: conversation_word_count, dtype: float64


### 1.5 Dataset Exploration: Missing Value Summary

A quick check for null/missing values across all columns before proceeding further.

In [8]:
missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values(
    'missing_count', ascending=False
)

if missing_summary.empty:
    print("No missing values found in any column.")
else:
    print(missing_summary)

No missing values found in any column.


**Initial observations:**
- Class imbalance present (~4x between largest and smallest class) — will need 
  per-class metrics rather than relying on plain accuracy later.
- Minimum conversation length is only 8 words, which is suspicious for a support 
  transcript — worth inspecting for truncated/corrupted rows in next part.
- No missing values in any column.

### 1.6 Data Quality Deep-Dive: Short Conversations (Day 4)

On Day 3, text length analysis flagged a suspicious minimum of 8 words in the `conversation` column — far below the median of ~358 words. Before proceeding, we inspect these short conversations directly to determine whether they represent legitimate (but brief) support interactions, truncated/corrupted data, or placeholder text that should be excluded.

In [9]:
# Sort by conversation word count and inspect the shortest examples
df_clean['word_count'] = df_clean['conversation'].str.split().str.len()

shortest = df_clean.nsmallest(10, 'word_count')[['issue_area', 'word_count', 'conversation']]

for idx, row in shortest.iterrows():
    print(f"--- Index {idx} | issue_area: {row['issue_area']} | words: {row['word_count']} ---")
    print(row['conversation'])
    print()

--- Index 196 | issue_area: Order | words: 8 ---
Agent: You're welcome, Jane. Have a great day!

--- Index 296 | issue_area: Order | words: 8 ---
Agent: You're welcome, Jane. Have a great day!

--- Index 771 | issue_area: Order | words: 8 ---
Agent: You're welcome, Jane. Have a great day!

--- Index 427 | issue_area: Order | words: 129 ---
Agent: Hello, thank you for contacting BrownBox customer support. My name is Alex. How can I assist you today? Customer: Hi Alex, I recently ordered a printer from your website and I wanted to check the order confirmation and status. Agent: Sure, I can definitely help you with that. Can you please provide me with your order number? Customer: Yes, it's #987654321. Agent: Thank you. I can see that the order has been confirmed, and the printer has been shipped. You should receive it within the next 2-3 business days. Is there anything else I can help you with? Customer: No, that's all. Thank you for your help. Agent: You're welcome. If you have any furt

In [10]:
# Cross-check: are these 3 truncated rows part of the "2 duplicates" found on Day 2?
truncated_idx = [196, 296, 771]

print("Full row comparison for the 3 truncated conversations:")
print(df_clean.loc[truncated_idx].to_string())

print("\n--- Duplicate check across ALL columns for these rows ---")
print(df_clean.loc[truncated_idx].duplicated(keep=False))

print("\n--- How many rows total have this exact conversation text? ---")
exact_match_count = (df_clean['conversation'] == "Agent: You're welcome, Jane. Have a great day!").sum()
print(f"Rows with this exact truncated text: {exact_match_count}")

print("\n--- Are there OTHER truncated/very short rows we haven't seen? ---")
print(df_clean[df_clean['word_count'] <= 15][['issue_area', 'issue_category', 'word_count']])

Full row comparison for the 3 truncated conversations:
    issue_area         issue_category                              issue_sub_category                                              issue_category_sub_category customer_sentiment product_category product_sub_category issue_complexity agent_experience_level                                                                                                                                                                                                                 agent_experience_level_desc                                    conversation  word_count
196      Order  Order Delivery Issues  Package shows as delivered but cannot be found  Order Delivery Issues -> Package shows as delivered but cannot be found         frustrated   Men/Women/Kids               Shorts             less                 junior                                                                                           handles customer inquiries independently, posse

### 1.7 Data Quality Deep-Dive: Short Conversations

Day 3's text length analysis flagged a suspicious minimum of 8 words. Inspecting the 10 shortest conversations shows the true minimum is not a legitimate short exchange — it's a truncation bug affecting 3 rows (indices 196, 296, 771), all `issue_area = Order`, which contain only the identical closing fragment *"Agent: You're welcome, Jane. Have a great day!"* rather than a full conversation.

A full-column comparison confirms the metadata (issue sub-category, product, sentiment, agent experience level) differs across all 3 rows — these are three distinct underlying tickets, but their conversation text failed to generate correctly. This also explains the "2 duplicate conversation texts" flagged during Day 2 cleaning: 3 identical strings register as 1 original + 2 duplicates under a single-column duplicate check.

All other conversations at or above 129 words are complete, legitimate agent/customer exchanges and are retained as-is.

**Action:** the 3 truncated rows are dropped, since `conversation` is the primary feature for classification and a row containing only a sign-off fragment carries no usable signal — retaining it would map an uninformative, repeated string to different labels and inject noise rather than help the model.

In [11]:
# Drop the 3 truncated rows identified above — no usable conversation signal
truncated_idx = [196, 296, 771]
df_clean = df_clean.drop(index=truncated_idx).reset_index(drop=True)

print(f"Rows after dropping truncated conversations: {len(df_clean)}")
print(f"New minimum word count: {df_clean['word_count'].min()}")

assert len(df_clean) == 997, f"Expected 997 rows after drop, got {len(df_clean)}"

df_clean.to_parquet("../data/conversations_clean.parquet", index=False)
print(f"Saved corrected dataset ({len(df_clean)} rows) to conversations_clean.parquet")

Rows after dropping truncated conversations: 997
New minimum word count: 129
Saved corrected dataset (997 rows) to conversations_clean.parquet


### 1.8 Sample Conversations by Issue Area

To get a qualitative feel for the data alongside the quantitative summaries above, we sample one representative conversation from each of the 6 `issue_area` classes. This helps confirm the labels are sensible (i.e. the conversation content actually matches its assigned category) before building any preprocessing or modeling pipeline on top of it.

In [12]:
# Pull one sample conversation per issue_area class for a qualitative sanity check
import textwrap

for area in sorted(df_clean['issue_area'].unique()):
    sample = df_clean[df_clean['issue_area'] == area].sample(1, random_state=42).iloc[0]
    print(f"=== issue_area: {area} ===")
    print(f"issue_category: {sample['issue_category']}")
    print(f"customer_sentiment: {sample['customer_sentiment']}")
    print(textwrap.fill(sample['conversation'], width=100))
    print()

=== issue_area: Cancellations and returns ===
issue_category: Return and Exchange
customer_sentiment: neutral
Agent: Thank you for calling BrownBox Customer Support. My name is Alex. How may I assist you today?
Customer: Hi, Alex. I recently purchased a Diaper from your website, but it's not the right size. I
would like to return or exchange it. Agent: I'm sorry to hear that. May I know your order number and
the reason for return or exchange? Customer: Sure, my order number is BB123456. I need to exchange
it for a larger size. Agent: Thank you for providing the details. I see that you are eligible for a
return or exchange. To initiate the process, I need to transfer you to our Returns and Exchanges
team. Please stay on the line while I transfer your call. [Agent transfers the call to the Returns
and Exchanges team] Returns and Exchanges team: Thank you for calling BrownBox Returns and
Exchanges. My name is Rachel. How may I assist you today? Customer: Hi, Rachel. I purchased a Diaper
f

### 1.9 Data Quality Issues — Summary (End of phase 1)

This section consolidates data quality findings from initial inspection through the deep-dive above.

**1. Missing values:** None. All 11 columns are fully populated across all rows.

**2. Duplicate rows:** No fully duplicated rows (all columns identical). However, 2 rows were flagged as having duplicate `conversation` text under a single-column check — this is explained below.

**3. Truncated conversations (resolved):** 3 rows (indices 196, 296, 771) contained only the fragment *"Agent: You're welcome, Jane. Have a great day!"* instead of a full conversation — a generation/truncation bug, not real short exchanges. All other metadata (issue sub-category, product, sentiment, agent experience) differed across the 3 rows, confirming these are 3 distinct tickets that each failed to generate conversation text correctly, not one ticket duplicated three times. This also explains the "2 duplicate conversation texts" flagged during Day 2 cleaning: 3 identical strings register as 1 original + 2 duplicates under `duplicated()`. These 3 rows were dropped, leaving **997 rows**. The new minimum conversation length is 129 words, consistent with the rest of the dataset.

**4. Whitespace in `issue_sub_category`:** Identified during Day 2 cleaning and corrected via `.str.strip()`.

**5. Class imbalance:** `issue_area` shows ~4x imbalance (Cancellations and returns ~28.6% vs. Shipping ~7.2%). This motivates using macro-averaged F1 (not just accuracy) when evaluating the classifier later, so minority classes aren't masked by majority-class performance.

**6. Label sanity check:** One sample conversation per `issue_area` class was manually inspected. In all 6 cases, the conversation content matched its assigned label (e.g. the "Cancellations and returns" sample was genuinely about an exchange; the "Warranty" sample was genuinely about warranty terms). No mislabeling patterns were observed in this spot check.

**Conclusion:** After removing the 3 truncated rows, the dataset (997 rows) is clean, non-null, free of full duplicates, and its labels are qualitatively sound. The main structural consideration going forward is class imbalance, which will inform metric choice and possibly class-weighting during modeling.

## Phase 2 — Preprocessing & Feature Engineering (Days 5–6)

### 2.1 Text Cleaning Pipeline

Before we can vectorize the conversation text, we need to normalize it. Raw transcripts contain
casing inconsistencies, punctuation, and common "stopwords" (e.g. "the", "is", "and") that add
noise without adding predictive signal for classifying `issue_area`.

Our cleaning pipeline applies the following steps, in order, to each conversation:

1. **Lowercase** — so "Shipping" and "shipping" aren't treated as different tokens.
2. **Remove punctuation** — punctuation marks aren't meaningful features for a bag-of-words/TF-IDF
   model in this context.
3. **Tokenize** — split the text into individual words.
4. **Remove stopwords** — filter out common English words that carry little topical meaning.
5. **Lemmatize** — reduce words to their dictionary/root form (e.g. "returned" → "return",
   "shipping" → "shipping", "issues" → "issue"), so inflected forms of the same word aren't
   treated as separate features.

We keep the full Agent+Customer transcript (rather than isolating just the customer's lines),
since the dataset doesn't provide structured turn-by-turn separation beyond the raw text, and
agent language (e.g. "your refund", "tracking number") is often just as predictive of `issue_area`
as customer language.

The cleaned output is stored in a new column, `conversation_clean`, leaving the original
`conversation` column untouched for reference/display purposes later in the notebook.

In [13]:
import re
import string
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# One-time downloads for NLTK's tokenizer, stopword list, and lemmatizer data.
# If you've already run these once, NLTK caches them locally and skips re-downloading.
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """
    Cleans and normalizes a single conversation string:
    lowercase -> remove punctuation -> tokenize -> remove stopwords -> lemmatize.
    Returns the cleaned tokens rejoined into a single string.
    """
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation (keep only letters, numbers, and whitespace)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Tokenize
    tokens = word_tokenize(text)

    # 4. Remove stopwords (and any leftover non-alphabetic tokens, e.g. stray numbers)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]

    # 5. Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)

# Load the cleaned dataframe from Day 4
df = pd.read_parquet('../data/conversations_clean.parquet')

# Apply the pipeline to the full dataset
df['conversation_clean'] = df['conversation'].apply(clean_text)

# Quick sanity check: compare a raw vs. cleaned example
print("RAW:\n", df['conversation'].iloc[0][:300])
print("\nCLEANED:\n", df['conversation_clean'].iloc[0][:300])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Paulflores\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Paulflores\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Paulflores\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Paulflores\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Paulflores\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


RAW:
 Agent: Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today? Customer: Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG), but I'm unable to proceed as it's asking for mobile number or email verification. Can you help me with tha

CLEANED:
 agent thank calling brownbox customer support name tom may assist today customer hi tom im trying log account purchase oven toaster grill otg im unable proceed asking mobile number email verification help agent sure assist may know registered mobile number email address please customer registered mo


Applied to all 997 conversations; verified with a raw-vs-cleaned example above.

In [14]:
df.to_parquet('../data/conversations_preprocessed.parquet')

### 2.2 Feature Engineering: TF-IDF Vectorization

We convert the cleaned conversation text into numerical features using **TF-IDF
(Term Frequency–Inverse Document Frequency)** rather than raw Bag-of-Words counts
or word embeddings (e.g. word2vec).

**Why TF-IDF over Bag-of-Words:** Our cleaned transcripts include speaker-label
tokens ("agent", "customer") that appear in nearly every row. Under plain word
counts, these tokens would dominate the feature space without carrying any
class-discriminative signal. TF-IDF's inverse-document-frequency term
automatically downweights words that appear across most documents, letting
genuinely distinctive words (e.g. "refund", "damaged", "tracking") carry more
weight for classification.

**Why TF-IDF over word embeddings:** With only 997 conversations, there isn't
enough data to train reliable embeddings from scratch, and pretrained embeddings
edge toward the deep-learning approaches this project intentionally avoids.
TF-IDF is also directly interpretable — each feature maps to an actual word,
which will matter later when we inspect top predictive features per class.

We cap vocabulary size with `max_features` and require a minimum document
frequency (`min_df`) to filter out one-off typos or noise tokens, keeping the
feature space manageable for a dataset this size.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# max_features caps vocab size to keep the matrix manageable for ~1000 rows
# min_df=2 drops words that appear in only 1 document (likely noise/typos)
tfidf = TfidfVectorizer(max_features=3000, min_df=2)

X_tfidf = tfidf.fit_transform(df["conversation_clean"])

print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

sparsity = 100 * (1.0 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]))
print(f"Sparsity: {sparsity:.2f}%")

TF-IDF matrix shape: (997, 1921)
Vocabulary size: 1921
Sparsity: 94.97%


## Phase 3 — Model & Metrics (Days 7–9)

### 3.1 Train/Test Split

Before training any classifier, we split the data into training and test sets.
The model will only ever see the training set during fitting; the test set is
held out entirely and used solely to evaluate performance on unseen data,
giving an honest estimate of how the model would perform on a new support
ticket.

We use `stratify=y` to preserve the ~4x class imbalance in `issue_area`
across both splits. Without stratification, a random split risks
under-representing minority classes in either the training set (hurting the
model's ability to learn that class) or the test set (making evaluation on
that class unreliable due to too few examples).

We use an 80/20 split — a standard default that leaves enough data to train
on while still holding out a reasonably sized test set, appropriate for a
dataset of this size (997 rows).

We train three models on this identical split, in increasing order of
sophistication, so each one can be read as an answer to the previous one's
weaknesses rather than an arbitrary list.

In [16]:
from sklearn.model_selection import train_test_split

# Target labels
y = df["issue_area"]

# 80/20 split, stratified to preserve class proportions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"\nTrain class distribution:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTest class distribution:\n{y_test.value_counts(normalize=True).round(3)}")

Train shape: (797, 1921)
Test shape: (200, 1921)

Train class distribution:
issue_area
Cancellations and returns    0.287
Order                        0.267
Login and Account            0.152
Shopping                     0.117
Warranty                     0.105
Shipping                     0.072
Name: proportion, dtype: float64

Test class distribution:
issue_area
Cancellations and returns    0.285
Order                        0.270
Login and Account            0.150
Shopping                     0.115
Warranty                     0.105
Shipping                     0.075
Name: proportion, dtype: float64


### 3.2 Baseline Model: Multinomial Naive Bayes

We start with **Multinomial Naive Bayes** as our baseline — the simplest
classic text-classification model. It estimates class probabilities directly
from word-frequency counts, assuming each word's occurrence is independent
of the others given the class. This is a strong assumption (word order and
co-occurrence clearly matter in real conversations), but it makes NB fast,
easy to reason about, and a natural floor to compare everything else against.

NB also has no `class_weight` parameter, so — unlike the two models that
follow — it receives no correction for the ~4x class imbalance in
`issue_area` established in Phase 1. This is intentional at the baseline
stage: it lets us see what imbalance-blind performance looks like before we
start correcting for it.

In [17]:
from sklearn.naive_bayes import MultinomialNB

nb_clf = MultinomialNB()
nb_clf.fit(X_train, y_train)

nb_train_acc = nb_clf.score(X_train, y_train)
nb_test_acc = nb_clf.score(X_test, y_test)

print(f"Naive Bayes — Train accuracy: {nb_train_acc:.3f}")
print(f"Naive Bayes — Test accuracy: {nb_test_acc:.3f}")

Naive Bayes — Train accuracy: 0.812
Naive Bayes — Test accuracy: 0.730


### 3.3 Second Model: Logistic Regression

Our second model is Logistic Regression — a discriminative linear classifier
that, unlike Naive Bayes, doesn't assume word independence and directly
optimizes for separating classes rather than modeling word-frequency
distributions per class. It's a standard, fast, interpretable step up from
NB for TF-IDF text classification, and its coefficients map directly to
per-word, per-class weights, which will matter later (Day 10, Predictive
Feature Analysis) when we look at which words drive each class's
predictions.

Critically, we set `class_weight="balanced"` here — directly addressing the
one-sided disadvantage NB was working under. If Logistic Regression beats NB,
we won't be able to tell from accuracy alone whether that's because it's a
better-suited model for text, or simply because it got imbalance correction
and NB didn't. Untangling that is part of why we look at macro-F1 in Section
3.5.

In [18]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

train_acc = clf.score(X_train, y_train)
test_acc = clf.score(X_test, y_test)

print(f"Logistic Regression — Train accuracy: {train_acc:.3f}")
print(f"Logistic Regression — Test accuracy: {test_acc:.3f}")

print(f"\nComparison so far:")
print(f"{'Model':<25}{'Train Acc':<12}{'Test Acc':<12}")
print(f"{'Naive Bayes':<25}{nb_train_acc:<12.3f}{nb_test_acc:<12.3f}")
print(f"{'Logistic Regression':<25}{train_acc:<12.3f}{test_acc:<12.3f}")

Logistic Regression — Train accuracy: 0.955
Logistic Regression — Test accuracy: 0.920

Comparison so far:
Model                    Train Acc   Test Acc    
Naive Bayes              0.812       0.730       
Logistic Regression      0.955       0.920       


### 3.4 Third Model: Linear SVM

Our third and final candidate is a **Linear Support Vector Machine**
(`LinearSVC`). Where Logistic Regression finds a probabilistic decision
boundary, a linear SVM finds the boundary that maximizes the margin between
classes — a different optimization objective that tends to handle wide,
sparse feature spaces like TF-IDF particularly well. It's a natural final
step: NB gave us an imbalance-blind floor, Logistic Regression showed what
imbalance correction alone buys us, and SVM tests whether a different
decision-boundary approach (with the same `class_weight="balanced"`
correction) can improve on that further.

We keep `class_weight="balanced"` and `random_state=42` consistent with
Logistic Regression so any difference between the two reflects the model
choice itself, not a difference in setup.

In [19]:
from sklearn.svm import LinearSVC

svm_clf = LinearSVC(class_weight="balanced", random_state=42)
svm_clf.fit(X_train, y_train)

svm_train_acc = svm_clf.score(X_train, y_train)
svm_test_acc = svm_clf.score(X_test, y_test)

print(f"Linear SVM — Train accuracy: {svm_train_acc:.3f}")
print(f"Linear SVM — Test accuracy: {svm_test_acc:.3f}")

print(f"\nFull comparison:")
print(f"{'Model':<25}{'Train Acc':<12}{'Test Acc':<12}")
print(f"{'Naive Bayes':<25}{nb_train_acc:<12.3f}{nb_test_acc:<12.3f}")
print(f"{'Logistic Regression':<25}{train_acc:<12.3f}{test_acc:<12.3f}")
print(f"{'Linear SVM':<25}{svm_train_acc:<12.3f}{svm_test_acc:<12.3f}")

Linear SVM — Train accuracy: 0.995
Linear SVM — Test accuracy: 0.945

Full comparison:
Model                    Train Acc   Test Acc    
Naive Bayes              0.812       0.730       
Logistic Regression      0.955       0.920       
Linear SVM               0.995       0.945       


### 3.5 Metrics Beyond Accuracy

The three models above are already ranked by accuracy, but accuracy alone can
be actively misleading on an imbalanced dataset. With `Shipping` at only
7.2% of the data, a model could score well overall while completely ignoring
that class — simply by leaning on the two largest classes (`Cancellations
and returns`, `Order`).

**Why not just use accuracy?** Accuracy is `correct predictions / total
predictions`. It weights every *ticket* equally — but with 4x more
`Cancellations and returns` tickets than `Shipping` tickets, that's the same
as weighting the `Cancellations and returns` *class* roughly 4x more heavily
than `Shipping` when judging the model. A model that nails the big classes
and whiffs on `Shipping` can still post a good accuracy score.

**Why not weighted-average F1 instead?** `classification_report`'s
weighted-average F1 computes F1 per class, then averages those scores
weighted by how many examples each class has (`support`). This is closer to
accuracy's per-ticket view than to a per-class view — it still lets a large
class's performance dominate the average and can mask a minority class's
failure almost as effectively as plain accuracy does. It answers "how well
does this model perform across all tickets," not "how well does this model
perform across all issue types."

**Why not micro-average F1?** For a single-label, multi-class problem like
ours (each ticket has exactly one `issue_area`), micro-averaged F1 reduces
mathematically to the same value as accuracy. It offers no additional
information over the accuracy numbers we already have above.

**Why macro-averaged F1:** Macro-F1 computes F1 independently for each of
the 6 classes, then averages those 6 scores with equal weight — regardless
of whether a class has 286 examples or 72. This directly matches what we
care about here: whether the model handles *every issue type* reasonably
well, not just the popular ones. A model that scores 0.99 F1 on the two
largest classes and 0.00 F1 on `Shipping` would still post a respectable
weighted-F1 or accuracy, but its macro-F1 would collapse — exactly the
failure mode we need visible before picking a final model.

We report macro-F1 as our primary metric below, alongside the full
per-class `classification_report` (precision/recall/F1/support) so we can
see *which* classes drive any gap between accuracy and macro-F1, not just
that a gap exists. We also generate a confusion matrix for the final model
to see exactly which classes get confused with each other, not just how
often the model is right or wrong overall.

In [20]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import plotly.express as px
import numpy as np

models = {
    "Naive Bayes": nb_clf,
    "Logistic Regression": clf,
    "Linear SVM": svm_clf,
}

macro_f1_scores = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    macro_f1_scores[name] = macro_f1

    print(f"\n{'='*60}")
    print(f"{name} — Macro F1: {macro_f1:.3f}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred, zero_division=0))

print("\nFinal Comparison")
print(f"{'Model':<25}{'Test Accuracy':<16}{'Macro F1':<12}")
print("-" * 53)
print(f"{'Naive Bayes':<25}{nb_test_acc:<16.3f}{macro_f1_scores['Naive Bayes']:<12.3f}")
print(f"{'Logistic Regression':<25}{test_acc:<16.3f}{macro_f1_scores['Logistic Regression']:<12.3f}")
print(f"{'Linear SVM':<25}{svm_test_acc:<16.3f}{macro_f1_scores['Linear SVM']:<12.3f}")


Naive Bayes — Macro F1: 0.593
                           precision    recall  f1-score   support

Cancellations and returns       0.67      0.98      0.80        57
        Login and Account       0.91      0.97      0.94        30
                    Order       0.65      0.81      0.72        54
                 Shipping       0.00      0.00      0.00        15
                 Shopping       1.00      0.26      0.41        23
                 Warranty       1.00      0.52      0.69        21

                 accuracy                           0.73       200
                macro avg       0.70      0.59      0.59       200
             weighted avg       0.72      0.73      0.68       200


Logistic Regression — Macro F1: 0.921
                           precision    recall  f1-score   support

Cancellations and returns       0.92      0.95      0.93        57
        Login and Account       1.00      1.00      1.00        30
                    Order       0.92      0.83      0.8

### 3.6 Metrics Summary & Final Model Selection

| Model | Test Accuracy | Macro F1 |
|---|---|---|
| Naive Bayes | 0.730 | 0.593 |
| Logistic Regression | 0.920 | 0.921 |
| Linear SVM | 0.945 | 0.947 |

Read as a progression, this table tells a coherent story rather than just
ranking three unrelated models:

- **Naive Bayes → Logistic Regression** is the biggest jump (macro-F1 0.593
  → 0.921), and it isn't purely an "algorithm quality" gap. Naive Bayes
  scores **0.00 precision/recall/F1 on `Shipping`** — the smallest class
  (7.2% of the data) — it never predicts that class correctly, despite a
  "reasonable-looking" 0.730 overall accuracy. It also struggles on
  `Shopping` (F1 0.41) and `Warranty` (F1 0.69). This is a direct
  consequence of NB having no `class_weight` correction, combined with its
  independence assumption being a poor fit for conversational text where
  phrasing and word co-occurrence carry real signal. Logistic Regression
  fixes both: imbalance correction via `class_weight="balanced"`, and a
  decision boundary that doesn't assume word independence.

- **Logistic Regression → Linear SVM** is a smaller, more incremental gain
  (macro-F1 0.921 → 0.947), consistent with both models sharing the same
  imbalance correction and differing only in *how* they draw the decision
  boundary. SVM's margin-maximizing objective edges out LogReg's
  probabilistic one on this high-dimensional, sparse TF-IDF feature space.

**Linear SVM is selected as the final model.** It has the highest accuracy
(0.945) and macro-F1 (0.947), and — importantly — the two scores are nearly
identical, indicating balanced performance across all 6 classes rather than
strength concentrated in the majority classes. Its weakest class (`Order`,
F1 0.91) is still solid, and it correctly identifies 100% of `Shipping`
tickets in the test set (recall 1.00) — the exact class Naive Bayes failed
on completely at the start of this progression.

One caution worth flagging: SVM's train accuracy (0.995) is notably higher
than its test accuracy (0.945), a larger train/test gap than Logistic
Regression's (0.955 → 0.920). This is a mild overfitting signal — SVM fits
the training data almost perfectly, which is expected given ~1,921 TF-IDF
features on only 797 training rows. Test performance still held up well,
but this gap is worth revisiting in the Reflections section.

In [21]:
# Confusion matrix for the selected final model (Linear SVM)
y_pred_svm = svm_clf.predict(X_test)
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred_svm, labels=labels)

fig = px.imshow(
    cm,
    x=labels,
    y=labels,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    text_auto=True,
    color_continuous_scale="Blues",
    title="Confusion Matrix — Linear SVM (Final Model)"
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

The confusion matrix shows a clean diagonal overall, with `Login and Account`
and `Shipping` classified perfectly (0 errors each) — both likely have
distinctive vocabulary that doesn't overlap with other categories. The small
number of errors that do occur cluster around `Order`: 2 `Cancellations and
returns` tickets and 2 `Shopping` tickets were misclassified as `Order`, and
`Order` itself was split across `Shipping` (3), `Cancellations and returns`
(2), and `Shopping` (1). This makes intuitive sense — order status,
cancellations, and shipping questions naturally share vocabulary in real
conversations (e.g. "where is my order" touches both `Order` and
`Shipping`), so some overlap in this region is expected rather than a
modeling flaw.

## Phase 4 — Predictive Feature Analysis (Day 10)

### 4.1 Top Predictive Features per Class

`LinearSVC` fits one linear decision boundary per class (one-vs-rest under
the hood), and each boundary has a coefficient for every one of our 1,921
TF-IDF features. A large positive coefficient for a given word, for a given
class, means that word's presence pushes the model strongly toward
predicting that class. Because TF-IDF features map directly to actual
vocabulary (unlike, say, PCA components or embedding dimensions), we can
read these coefficients directly as "the words the model is relying on" for
each `issue_area` — this interpretability was one of the reasons we chose
TF-IDF over embeddings back in Section 2.2.

**A note on `agent`/`customer` tokens:** our cleaned conversations retain
speaker-label tokens like "agent" and "customer" (Section 2.1 keeps the full
transcript rather than isolating one speaker), and these words appear in
nearly every row regardless of `issue_area`. Under raw word counts, tokens
this common would swamp the top-features list without telling us anything
class-specific. TF-IDF's inverse-document-frequency term is exactly what
prevents that: a word that appears in almost every document gets a low IDF
weight, so "agent" and "customer" end up contributing little to the
TF-IDF score even though they're frequent. We'd expect the lists below to
be dominated by genuinely topical words (e.g. "refund," "warranty",
"tracking") rather than these near-universal speaker labels — which itself
is a useful sanity check on whether TF-IDF is doing its job.

For each of the 6 classes, we extract the 15 words with the largest positive
coefficient in that class's decision boundary.

In [22]:
import pandas as pd

feature_names = tfidf.get_feature_names_out()
classes = svm_clf.classes_  # order matches rows of svm_clf.coef_

top_n = 15
top_features_per_class = {}

for i, class_name in enumerate(classes):
    coefs = svm_clf.coef_[i]  # coefficient vector for this class vs. rest
    top_idx = np.argsort(coefs)[::-1][:top_n]  # indices of largest coefficients
    top_features_per_class[class_name] = [
        (feature_names[idx], round(coefs[idx], 3)) for idx in top_idx
    ]

# Display as a clean side-by-side table
top_features_df = pd.DataFrame({
    cls: [word for word, _ in feats]
    for cls, feats in top_features_per_class.items()
})
top_features_df.index = [f"#{i+1}" for i in range(top_n)]
top_features_df

,Cancellations and returns,Login and Account,Order,Shipping,Shopping,Warranty
#1,pickup,account,invoice,location,stock,warranty
#2,refund,verification,installation,faster,discount,claim
#3,returned,mobile,delivered,inaday,create,registration
#4,tampered,code,order,standard,price,detail
#5,return,deactivate,repair,charge,international,start
#6,replacement,password,payment,provider,different,date
#7,bank,address,executive,shipping,point,pendrive
#8,cancel,new,status,service,hidden,term
#9,cancellation,signup,mary,delivery,wall,visit
#10,received,otp,damaged,free,mount,purchased


### 4.2 What the Model Learned

**Class-word alignment is strong.** Each class's top words map clearly onto
its label, which is a good sanity check that the model is learning genuine
topical signal rather than noise:

- **Cancellations and returns:** `pickup`, `refund`, `returned`, `return`,
  `replacement`, `cancel`, `cancellation`, `exchange` — almost the entire
  list is directly on-topic vocabulary for this class.
- **Login and Account:** `account`, `verification`, `password`, `otp`,
  `signup`, `email`, `app` — clearly account-access language.
- **Order:** `invoice`, `installation`, `delivered`, `order`, `payment`,
  `status`, `billing` — general order-lifecycle terms.
- **Shipping:** `location`, `faster`, `provider`, `shipping`, `delivery`,
  `speed` — delivery-logistics vocabulary.
- **Shopping:** `stock`, `discount`, `price`, `cashback`, `offer` —
  pre-purchase/pricing language.
- **Warranty:** `warranty`, `claim`, `registration`, `term`, `date` —
  warranty-process vocabulary.

**The `agent`/`customer` prediction held.** As anticipated in Section 4.1,
neither speaker-label token appears in any of the 6 top-15 lists. TF-IDF's
IDF weighting worked as intended: these near-universal tokens were
downweighted enough that genuinely class-specific vocabulary dominates the
model's decision boundaries instead.

**A few unexpected entries are worth flagging.** `mary` (Order, #9) and
`maria` (Shipping, #12) are customer/agent first names, not topical words —
their presence suggests these particular names happen to co-occur with
their class more often in this synthetic dataset, purely by chance in how
GPT-3.5 generated the conversations, rather than reflecting any real-world
signal. Similarly, `canon` and `pendrive` (Warranty) and `treadmill` and
`ceiling` (Shopping, likely from "ceiling fan" or a wall-mount product) are
specific product names rather than issue-type vocabulary — with a synthetic
dataset built from a fixed pool of `product_sub_category` values, certain
products probably cluster disproportionately within certain issue types
(e.g. more warranty tickets happen to be about a Canon product than chance
alone would suggest at this dataset size). This is a useful limitation to
note: with only 997 rows split across 6 classes, some "predictive" words are
really artifacts of which specific products/names happened to appear in
which class's synthetic examples, not durable topical signal a production
model could rely on.

## Phase 5 — Reflections & Errors (Day 10)

### 5.1 Strengths

**Strong, well-separated performance for a ~1,000-row dataset.** The final
Linear SVM reaches 0.945 test accuracy and 0.947 macro-F1 across 6 classes —
and those two numbers being nearly identical (rather than accuracy
outpacing macro-F1) is itself a positive signal, since it means performance
isn't concentrated in the majority classes. The confusion matrix (3.6)
backs this up qualitatively: `Login and Account` and `Shipping` — the
smallest class — were classified with zero errors, and the small number of
mistakes that do occur cluster in a semantically sensible place (`Order`
overlapping with `Shipping`/`Cancellations and returns`), not scattered
randomly.

**Interpretability held up under inspection.** Because we chose TF-IDF over
embeddings specifically for interpretability (Section 2.2), Section 4's
top-feature analysis let us actually verify the model is learning real
topical vocabulary per class (`refund`/`return` for Cancellations,
`password`/`otp` for Login, etc.) rather than being a black box we just
trust because the accuracy number looks good.

### 5.2 Weaknesses

**Naive Bayes exposed the risk of skipping imbalance correction.** NB was
included specifically as an imbalance-blind baseline (3.2), and the result
was stark: 0.00 precision/recall/F1 on `Shipping`, the smallest class
(7.2% of the data), despite a deceptively reasonable-looking 0.730 overall
accuracy. This is the clearest evidence in the whole notebook for why
accuracy alone is an unreliable metric on imbalanced data, and why both
`class_weight="balanced"` and macro-F1 (rather than accuracy or
weighted-F1) were necessary design choices, not optional extras.

**Mild overfitting in the final model.** The selected Linear SVM shows a
noticeably larger train/test gap (0.995 → 0.945) than Logistic Regression
(0.955 → 0.920), as flagged in 3.6. This is expected given ~1,921 TF-IDF
features fit on only 797 training rows — a high-dimensional, sparse feature
space relative to dataset size gives the model room to fit training-set
idiosyncrasies. Test performance still held up well, but this gap means the
0.945 test accuracy is likely a slight overestimate of how the model would
perform on a genuinely new batch of tickets from outside this dataset.

**Some "predictive" features are likely dataset artifacts, not durable
signal.** Section 4.2 found customer names (`mary`, `maria`) and specific
product names (`canon`, `treadmill`) among the top features for particular
classes. With only ~1,000 synthetically generated conversations, these are
almost certainly coincidental co-occurrences from how the dataset was
generated, not real-world signal a production model could rely on. This is
a direct consequence of dataset size — a larger, more diverse training set
would likely dilute these artifacts.

### 5.3 Class Imbalance — Summary

`issue_area` carries a ~4x imbalance between the largest class
(`Cancellations and returns`, 28.6%) and the smallest (`Shipping`, 7.2%),
identified during Phase 1 exploration (Section 1.3). This shaped three
separate modeling decisions: using `stratify=y` in the train/test split
(3.1) so both sets reflect the same class proportions, using
`class_weight="balanced"` in Logistic Regression and SVM (3.3, 3.4) so the
loss function doesn't get dominated by majority classes, and reporting
macro-F1 as the primary metric (3.5) so per-class performance — not just
overall correctness — determines model selection. Naive Bayes' failure on
`Shipping` is the clearest demonstration of what happens when none of these
three corrections are applied.

### 5.4 What Would Improve This Further

Beyond the scope of this notebook, a few directions could push performance
or robustness further: a larger dataset would reduce the risk of
coincidental artifacts like those found in Section 4.2 and give minority
classes like `Shipping` more examples to learn from; hyperparameter tuning
(e.g. grid search over SVM's `C` or TF-IDF's `max_features`/`min_df`) could
recover some of the overfitting gap noted in 5.2; and — as a genuinely
different approach rather than a tuning tweak — pretrained sentence
embeddings could capture semantic similarity between phrasings that TF-IDF's
exact-word-match features can't (e.g. "my package never arrived" vs. "I
never got my delivery" would score as more similar under embeddings than
under TF-IDF). That's intentionally outside this project's scope, which
sticks to classic ML per the course list, but it's a natural next step for
a production version of this classifier.